In [ ]:
using FastGaussQuadrature
using SpecialFunctions
using Plots

function gen_laguerre(n::Int, alpha::Float64, x::Float64)
    if n == 0 return 1.0 end
    if n == 1 return 1.0 + alpha - x end
    
    L_prev = 1.0
    L_curr = 1.0 + alpha - x
    
    for k in 1:(n-1)
        L_next = ((2*k + 1 + alpha - x) * L_curr - (k + alpha) * L_prev) / (k + 1)
        L_prev = L_curr
        L_curr = L_next
    end
    return L_curr
end

function hermite_laguerre_to_xmax(rawNodes, rawWeights, data, xmax, x_eval, T)
    
    scale = xmax / maximum(rawNodes)
    n = length(rawNodes)
    
    xi_std = rawNodes ./ sqrt(T)
    
    function get_norm(k)
        return sqrt((scale * sqrt(T)) * gamma(k + 0.5) / factorial(k))
    end
    
    coeffs = Float64[]
    for m in 0:(n-1)
        c = 0.0
        norm_m = get_norm(m)
        
        for i in 1:n
            term = data[i]
            term *= exp(xi_std[i]^2)
            term *= gen_laguerre(m, -0.5, xi_std[i]^2)
            term *= (rawWeights[i] * scale) / norm_m
            c += term
        end
        push!(coeffs, c)
    end
    arg_x = (x_eval / (scale * sqrt(T)))^2
    
    poly_part = 0.0
    for i in 0:(n-1)
        poly_part += coeffs[i+1] * gen_laguerre(i, -0.5, arg_x) / get_norm(i)
    end
    
    return poly_part * exp(-arg_x)
end


In [ ]:
p = plot()
for i in 0:5
    f(x) = x*gen_laguerre(i, 1/2, x^2)*exp(-x^2)
    plot!(f,0,5)
end
display(p)


In [ ]:

n = 8
T = 2.0
xmax = 6.0

nodes_all, weights_all = gausshermite(2*n)
xi_std = nodes_all[n+1:end]       
wi_std = weights_all[n+1:end] .* 2 

raw_nodes = xi_std .* sqrt(T)
raw_weights = wi_std .* sqrt(T)

f(x) = (1+x^2 + x^4) * exp(-x^2/T)

current_scale = xmax / maximum(raw_nodes)
sampling_points = raw_nodes .* current_scale
yi = f.(sampling_points)

x_plot = range(0, xmax + 1, length=200)

y_true = f.(x_plot)
y_recon = [hermite_laguerre_to_xmax(raw_nodes, raw_weights, yi, xmax, x, T) for x in x_plot]

plot(x_plot, y_true, 
    label="f(x) (Independent)", 
    lw=2, color=:blue, 
    title="Interpolation with xmax = $xmax")

plot!(x_plot, y_recon, 
    label="Reconstruction", 
    ls=:dash, lw=2, color=:red)

scatter!(sampling_points, yi, 
    label="Sampling Nodes", 
    color=:black, marker=:circle)

In [ ]:
using Plots, bslLD


n = 8
T = 2.0
xmax = 6.0

nodes_all, weights_all = gausshermite(2*n)
xi_std = nodes_all[n+1:end]       
wi_std = weights_all[n+1:end] .* 2 

raw_nodes = xi_std .* sqrt(T)
raw_weights = wi_std .* sqrt(T)

current_scale = xmax / maximum(raw_nodes)
sampling_points = raw_nodes .* current_scale


vp = sampling_points

nphi = 10
phi = 0:2pi/nphi:(nphi-1)/nphi*2*pi
finit1(r, alpha) = exp(-r^2/2) *(1+sin(alpha)*r^2 + cos(2*alpha)*r^4 * cos(5*alpha)+r^3)


dist = [finit1(vpi, phii) for vpi in vp, phii in phi]
println(size(dist))

imax = 5
F = bslLD.fft(dist, [2])/(2*pi)
p = plot(layout = (3,2), size = (1000, 1000),left_margin = 12Plots.mm)


heatmap!(p[1],phi, vp, dist; projection = :polar,)

x_plot = range(0, xmax + 1, length=200)
phi_plot = range(0, 2pi, length=200)



frecon = zeros(ComplexF64, length(x_plot), length(phi_plot))
for k in 0:round(Int64,nphi/2-1)
    for ipart in 1:2
        fft_idx = k + 1 
        m = k 
        
        Ck_r_nodes = F[:, fft_idx] 

        Ck_r_interp  = [hermite_laguerre_to_xmax(raw_nodes, raw_weights, (real, imag)[ipart].(Ck_r_nodes), xmax, x, T) for x in x_plot]
        
        mode_contribution = Ck_r_interp .* exp.(1im * m * phi_plot')
        
        if m == 0
            global frecon .+= mode_contribution
        else
            global frecon .+= 2 * real.(mode_contribution)
        end

        plot_idx = k + 3
        if plot_idx <= 6
                plot!(p[plot_idx], vp, real.(Ck_r_nodes), marker=:o, label="Nodes (Re)", title="Mode m=$m")
                plot!(p[plot_idx], x_plot, real.(Ck_r_interp), line=:dash, label="Recon (Re)")
                plot!(p[plot_idx], vp, imag.(Ck_r_nodes), marker=:x, label="Nodes (Im)")
                plot!(p[plot_idx], x_plot, imag.(Ck_r_interp), line=:dot, label="Recon (Im)")
        end
    end
end

heatmap!(p[2], phi_plot, x_plot, real.(frecon), projection=:polar)

display(p)

In [ ]:
vp